# Erosional sediment flux out of the Mataquito watershed (outlet CT-9)

## Overview

**Purpose**: Estimate the volumetric erosional sediment flux leaving the Mataquito watershed at the
outlet sample CT-9, by summing the fluxes of the domains that supply it.

- **Route A (primary)**: parent sum. Principal Cordillera parents (CT-5 + CT-6) + Central Depression
  (0, a transfer zone) + Coastal Cordillera canyon CT-3* (represented by CT-3).
- **Route B (check)**: un-weight the measured downstream rates (PC, then CT-9) by quartz fertility.
  Algebraically this reduces to Route A.
- **Single-sample budgets (comparison)**: S1 = outlet only; S2 = one downstream PC sample + CT-3*.

**Configurations**: the whole calculation is run twice, with a different downstream Principal
Cordillera reference sample (PC). PC sets where the Central Depression ends and CT-3* begins.

| configuration | PC sample | sample_data ID |
|---|---|---|
| `PC_CT10` | CT-10 | `CT-10` |
| `PC_CT82` | CT-8.2 | `CT-8` |

**Inputs**: `data/sample_metadata/MataquitoSampleData.xlsx` (via `mataquito.sample_data`). Table S7
Part B also reads the quartz and zircon flux ratios written by notebook 06 to `results/fertility/`.

**Outputs** (`results/mass_balance/`, one row per quantity per configuration):
- `mass_balance_summary.csv`: median, IQR, 2.5-97.5 percentile
- `area_accounting.csv`: domain areas
- `supplemental/Table_S7.xlsx`: sediment budget (PC_CT82) and mineral flux ratios (section 8)

**Units**: rates in m/Myr, areas in km², volumetric flux in m³/yr.
1 m/Myr × 1 km² = 1e-6 m/yr × 1e6 m² = **1 m³/yr**, so E [m/Myr] × A [km²] is already in m³/yr.

In [1]:
import os                      # path joining / (commented-out) figure directory creation
import sys                     # to put the package root on the import path
import numpy as np             # vectorised Monte Carlo arithmetic
import pandas as pd            # tabular outputs

# Make the local `mataquito` package importable (notebooks/ sits one level below it)
sys.path.insert(0, '..')
from mataquito.sample_data import load_samples, get_areas, get_production_rates
from mataquito.revised_erosion import generate_mc_samples, flux_order_samples
from mataquito.revised_fertility import quartz_fertility

# Absolute input/output locations. This notebook writes to results/mass_balance/ and supplemental/.
ROOT = '/Users/Glong1/Desktop/Andes/Chapter_1_Manuscript/MataquitoWatershedAnalysis/Mataquito'
OUT = os.path.join(ROOT, 'results', 'mass_balance')
FIG = os.path.join(ROOT, 'figures', 'mass_balance')   # used only by the commented-out figure cells
SUPP = os.path.join(ROOT, 'supplemental')             # manuscript supplemental tables; Table S7 is written here
FERT = os.path.join(ROOT, 'results', 'fertility')     # notebook 06 fertility CSVs, read for Table S7 Part B

# Monte Carlo settings, matching the manuscript Methods
N = 100_000
SEED = 17

# Configuration name -> sample_data ID of the downstream Principal Cordillera reference sample
CONFIGS = {'PC_CT10': 'CT-10', 'PC_CT82': 'CT-8'}   # CT-8.2 is stored as 'CT-8'

## 1. Samples, Monte Carlo draws and flux ordering

Erosion-rate draws are normal (mean = measured rate, sd = external uncertainty). Flux ordering follows
the manuscript Methods ("Flux-Ordered Erosion Rates"): CT-7 → CT-5, CT-1 → CT-6, then CT-5 + CT-6 →
each PC sample. Each `triples[PC]` is index-aligned, so draw *i* of CT-5, CT-6 and the PC sample
belongs to the same Monte Carlo iteration.

In [2]:
df = load_samples()                 # canonical sample spreadsheet
A = get_areas(df)                   # {Sample_ID: full watershed area, km^2}
P = get_production_rates(df)        # {Sample_ID: surface 10Be production rate, atoms/g/yr}

ER = generate_mc_samples(df, N=N, seed=SEED)          # {Sample_ID: N normal draws of E, m/Myr}
fo = flux_order_samples(ER, A, N=N, seed=SEED)        # apply the downstream-increasing-flux filter

# Fraction of draws surviving each ordering constraint (reported for transparency)
for k, v in fo['retained'].items():
    print(f'{k:28s} retained {v:6.1%}')

# Both PC children are filtered against the same resampled CT-5/CT-6 arrays. With 100% retention
# the parent draws are identical in both triples, which makes the configurations comparable draw by
# draw (needed for the cross-configuration check in section 6).
E5, E6, _ = fo['triples']['CT-10']
for pc in CONFIGS.values():
    assert np.array_equal(fo['triples'][pc][0], E5) and np.array_equal(fo['triples'][pc][1], E6), \
        f'CT-5/CT-6 draws in the {pc} triple are not the shared parent draws'
M = len(E5)                         # number of paired draws used throughout

# CT-3 and CT-9 carry no ordering constraint and are drawn independently of the CT-5/6/PC draws,
# so taking their first M draws gives a valid independent pairing of equal length.
E3 = fo['CT3_res'][:M]              # CT-3 (resampled, unconstrained)
E9 = ER['CT-9'][:M]                 # CT-9 outlet (raw draws; not flux-ordered)
print(f'\nPaired draws used throughout: M = {M:,}')

CT-7 -> CT-5                 retained  20.8%
CT-1 -> CT-6                 retained 100.0%
CT-5 + CT-6 -> CT-10         retained 100.0%
CT-5 + CT-6 -> CT-11         retained 100.0%
CT-5 + CT-6 -> CT-8          retained 100.0%
CT-4 (unconstrained)         retained 100.0%

Paired draws used throughout: M = 100,000


## 2. Area accounting

Full watershed areas are used (no Central Depression adjustment). For each configuration the four
pieces must tile CT-9 exactly:

- $A_{CD} = A(PC) - A(CT\text{-}5) - A(CT\text{-}6)$
- $A_{CC} = A(CT\text{-}9) - A(PC)$ (CT-3* for that configuration)

In [3]:
area_rows = []                       # long-format area table, one row per domain per configuration
AREAS = {}                           # {config: (A_PC_parents, A_CD, A_CC)} for the later sections
for cfg, pc in CONFIGS.items():
    A_par = A['CT-5'] + A['CT-6']                # Principal Cordillera parents (Teno + Lontué)
    A_CD = A[pc] - A['CT-5'] - A['CT-6']         # Central Depression between the parents and PC
    A_CC = A['CT-9'] - A[pc]                     # CT-3*: Coastal Cordillera residual below PC
    # The four pieces must sum to the outlet area (floating-point tolerance only)
    assert np.isclose(A['CT-5'] + A['CT-6'] + A_CD + A_CC, A['CT-9'], rtol=0, atol=1e-9), \
        f'{cfg}: domains do not tile CT-9'
    AREAS[cfg] = (A_par, A_CD, A_CC)
    for dom, defn, a in [('PC parents', 'A(CT-5) + A(CT-6)', A_par),
                         ('CD', f'A({pc}) - A(CT-5) - A(CT-6)', A_CD),
                         ('CC (CT-3*)', f'A(CT-9) - A({pc})', A_CC),
                         ('Total (CT-9)', 'A(CT-9)', A['CT-9'])]:
        area_rows.append({'configuration': cfg, 'domain': dom, 'definition': defn,
                          'area_km2': a, 'fraction_of_CT9': a / A['CT-9']})

area_tbl = pd.DataFrame(area_rows)
print(area_tbl.to_string(index=False))
area_tbl.to_csv(os.path.join(OUT, 'area_accounting.csv'), index=False)

configuration       domain                   definition  area_km2  fraction_of_CT9
      PC_CT10   PC parents            A(CT-5) + A(CT-6)  4069.142         0.657404
      PC_CT10           CD A(CT-10) - A(CT-5) - A(CT-6)   795.800         0.128568
      PC_CT10   CC (CT-3*)           A(CT-9) - A(CT-10)  1324.774         0.214028
      PC_CT10 Total (CT-9)                      A(CT-9)  6189.716         1.000000
      PC_CT82   PC parents            A(CT-5) + A(CT-6)  4069.142         0.657404
      PC_CT82           CD  A(CT-8) - A(CT-5) - A(CT-6)   881.253         0.142374
      PC_CT82   CC (CT-3*)            A(CT-9) - A(CT-8)  1239.321         0.200223
      PC_CT82 Total (CT-9)                      A(CT-9)  6189.716         1.000000


## 3-5. Route A, Route B and single-sample budgets, per configuration

**Route A**: $F_{PC} = E_5 A_5 + E_6 A_6$, $F_{CD} = 0$, $F_{CC} = E_3 A_{CC}$,
$F_{total} = F_{PC} + F_{CD} + F_{CC}$, $E_{mass} = F_{total} / A_9$. The PC rate is
$F_{PC} / (A_5 + A_6)$, the parent flux over the parent area only. The Central Depression is a
non-contributing transfer zone ($F_{CD} = 0$), so its area is excluded from the denominator.

**Route B**: a ¹⁰Be rate measured downstream of a confluence is **quartz-flux weighted**. Writing
nuclide concentration as $N \propto P/E$, Eq. 4 for parents A, B and child C (relative fertility
$q = Q_A/Q_B$, $Q_B \equiv 1$) gives

$$\frac{P_C}{E_C} = \frac{q A_A P_A + A_B P_B}{q E_A A_A + E_B A_B}
\;\Rightarrow\; F_Q \equiv q E_A A_A + E_B A_B = \frac{E_C\,(q A_A P_A + A_B P_B)}{P_C}.$$

So the measured child rate implies the total quartz-flux supply $F_Q$. `quartz_fertility()` also
returns the quartz flux ratio $R = F_{QA}/F_{QB}$, which gives each parent's quartz-flux fraction
$f_A = R/(1+R)$. Dividing each parent's quartz flux by its relative fertility converts it back to
volumetric flux:

$$F = F_Q\left[\frac{f_A}{q} + \frac{1 - f_A}{1}\right].$$

- **Step 1**: CT-5 & CT-6 → PC gives $q_1 = Q_{Teno}/Q_{Lontué}$ and $F_{PC}^{(B)}$.
- **Step 2**: PC & CT-3* → CT-9, with $A_A = A(PC)$ and $A_B = A_{CC}$. The CT-3* term is un-weighted
  the same way, and $F_{PC}^{(B)}$ replaces the PC term to give $F_{total}^{(B)}$.

Draws with an unphysical (negative) fertility are masked by `quartz_fertility()`, so Route B is
evaluated on the draws retained by **both** steps. If the Route A and Route B medians differ by more
than 1%, the notebook stops.

**Single-sample budgets**: S1 = $E_9 A_9$ (the same in both configurations);
S2 = $E_{PC} A(PC) + E_3 A_{CC}$.

In [4]:
def run_config(cfg, pc):
    # Full calculation for one PC reference sample; returns {quantity: (units, per-draw array)}
    A_par, A_CD, A_CC = AREAS[cfg]
    E_pc = fo['triples'][pc][2]              # PC draws, index-aligned with E5/E6

    # ---- Route A: parent sum ----
    F_PC = E5 * A['CT-5'] + E6 * A['CT-6']   # m^3/yr; each parent's rate times its own area
    F_CD = np.zeros(M)                       # transfer zone: no net erosional contribution
    F_CC = E3 * A_CC                         # CT-3 rate applied to this configuration's CT-3* area
    F_total = F_PC + F_CD + F_CC             # m^3/yr leaving CT-9
    E_mass = F_total / A['CT-9']             # area-normalised, mass-weighted basin rate, m/Myr
    # PC rate over the parent area A(CT-5) + A(CT-6), m/Myr. The Central Depression is a non-contributing
    # transfer zone (F_CD = 0), so its area is excluded from the denominator.
    E_PC_mass = F_PC / A_par

    # ---- Route B step 1: CT-5 (A, Teno) & CT-6 (B, Lontué) -> PC ----
    # q1 = Q_Teno/Q_Lontue, R1 = quartz flux ratio; both already masked to physical draws m1
    q1, R1, m1 = quartz_fertility(E5, E6, E_pc, A['CT-5'], A['CT-6'], P['CT-5'], P['CT-6'], P[pc])
    # Quartz-flux supply implied by the measured PC rate (Eq. 4), in Lontué-fertility units
    FQ1 = E_pc[m1] * (q1 * A['CT-5'] * P['CT-5'] + A['CT-6'] * P['CT-6']) / P[pc]
    fA1 = R1 / (1 + R1)                      # fraction of the PC quartz flux supplied by CT-5
    # Un-weight: divide each parent's quartz flux by its relative fertility (q1 Teno, 1 Lontué)
    F_PC_B1 = FQ1 * (fA1 / q1 + (1 - fA1))   # volumetric PC flux, m^3/yr (on draws m1)

    # ---- Route B step 2: PC (A) & CT-3* (B) -> CT-9, on the step-1 survivors ----
    q2, R2, m2 = quartz_fertility(E_pc[m1], E3[m1], E9[m1], A[pc], A_CC, P[pc], P['CT-3'], P['CT-9'])
    # Quartz-flux supply implied by the measured CT-9 rate, in CT-3*-fertility units
    FQ2 = E9[m1][m2] * (q2 * A[pc] * P[pc] + A_CC * P['CT-3']) / P['CT-9']
    fA2 = R2 / (1 + R2)                      # fraction of the CT-9 quartz flux supplied by PC
    F_CC_B = FQ2 * (1 - fA2)                 # CT-3* quartz flux / its fertility (1) -> volumetric flux
    F_PC_B = F_PC_B1[m2]                     # step-1 PC flux on the jointly retained draws
    F_total_B = F_PC_B + F_CC_B              # F_PC(B) as the PC term, plus CD = 0, plus CT-3*
    keep = np.flatnonzero(m1)[m2]            # indices (into length-M arrays) retained by both steps

    # ---- Route B check (1% stop rule) ----
    rel_diff = (F_total_B - F_total[keep]) / F_total[keep]   # per-draw relative difference
    d_same = np.median(F_total_B) / np.median(F_total[keep]) - 1   # medians on the same draws
    d_full = np.median(F_total_B) / np.median(F_total) - 1         # vs the full Route A ensemble
    print(f'{cfg}: Route B joint retention {len(keep)/M:.1%}; per-draw rel. diff median '
          f'{np.median(rel_diff):.2e} [{np.percentile(rel_diff, 2.5):.2e}, '
          f'{np.percentile(rel_diff, 97.5):.2e}]; median diff same draws {d_same:+.3%}, '
          f'full ensemble {d_full:+.3%}')
    if max(abs(d_same), abs(d_full)) > 0.01:
        raise RuntimeError(f'{cfg}: Route A and Route B medians differ by more than 1% '
                           f'(same draws {d_same:+.3%}, full ensemble {d_full:+.3%})')

    # ---- Single-sample budgets ----
    S1 = E9 * A['CT-9']                      # outlet rate applied to the whole basin, m^3/yr
    S2 = E_pc * A[pc] + E3 * A_CC            # PC rate for everything above CT-3*, m^3/yr

    return {
        'F_PC (A)': ('m3/yr', F_PC), 'F_CD (A)': ('m3/yr', F_CD), 'F_CC (A)': ('m3/yr', F_CC),
        'F_total (A)': ('m3/yr', F_total), 'E_mass (A)': ('m/Myr', E_mass),
        'E_PC_mass = F_PC/(A5+A6) (A)': ('m/Myr', E_PC_mass),
        'share_PC (A)': ('fraction', F_PC / F_total), 'share_CD (A)': ('fraction', F_CD / F_total),
        'share_CC (A)': ('fraction', F_CC / F_total),
        'q1 = Q_Teno/Q_Lontue (B)': ('ratio', q1),
        'F_PC (B)': ('m3/yr', F_PC_B), 'F_total (B)': ('m3/yr', F_total_B),
        'rel_diff (B-A)/A': ('fraction', rel_diff),
        'S1 = E9*A9': ('m3/yr', S1), 'S1 rate': ('m/Myr', S1 / A['CT-9']),
        'S2 = E_PC*A_PC + E3*A_CC': ('m3/yr', S2), 'S2 rate': ('m/Myr', S2 / A['CT-9']),
        'S1/F_total': ('ratio', S1 / F_total), 'S2/F_total': ('ratio', S2 / F_total),
    }

# Run every configuration and keep the per-draw arrays for the summary and cross-check
RES = {cfg: run_config(cfg, pc) for cfg, pc in CONFIGS.items()}
for cfg in CONFIGS:
    print(f"{cfg}: median F_total {np.median(RES[cfg]['F_total (A)'][1]):,.0f} m^3/yr; "
          f"F_PC/(A5+A6) {np.median(RES[cfg]['E_PC_mass = F_PC/(A5+A6) (A)'][1]):.1f} m/Myr")

PC_CT10: Route B joint retention 99.8%; per-draw rel. diff median 0.00e+00 [-3.41e-16, 3.46e-16]; median diff same draws +0.000%, full ensemble +0.014%
PC_CT82: Route B joint retention 98.4%; per-draw rel. diff median 0.00e+00 [-5.13e-16, 5.21e-16]; median diff same draws +0.000%, full ensemble +0.107%
PC_CT10: median F_total 734,589 m^3/yr; F_PC/(A5+A6) 170.9 m/Myr
PC_CT82: median F_total 732,074 m^3/yr; F_PC/(A5+A6) 170.9 m/Myr


## 6. Cross-configuration check

The configurations share every draw and differ only in where the 85.5 km² reach between CT-10 and
CT-8.2 is counted. Per draw, therefore:

$$F_{total}(\text{PC\_CT10}) - F_{total}(\text{PC\_CT82}) = E_3\,[A(CT\text{-}8) - A(CT\text{-}10)].$$

In [5]:
# Left side: per-draw difference of the two Route A totals
dF = RES['PC_CT10']['F_total (A)'][1] - RES['PC_CT82']['F_total (A)'][1]
# Right side: CT-3 rate applied to the reach CT-8.2 adds to the PC basin (CC in PC_CT10, CD in PC_CT82)
dF_expected = E3 * (A['CT-8'] - A['CT-10'])
assert np.allclose(dF, dF_expected, rtol=1e-9, atol=1e-6), 'cross-configuration identity fails'
print(f'A(CT-8) - A(CT-10) = {A["CT-8"] - A["CT-10"]:.1f} km^2; '
      f'median difference {np.median(dF):,.0f} m^3/yr '
      f'[{np.percentile(dF, 2.5):,.0f}, {np.percentile(dF, 97.5):,.0f}]; '
      f'max |residual| {np.max(np.abs(dF - dF_expected)):.1e} m^3/yr')

A(CT-8) - A(CT-10) = 85.5 km^2; median difference 2,520 m^3/yr [2,111, 2,925]; max |residual| 1.2e-10 m^3/yr


## 7. Summary table

In [6]:
def summarize(cfg, name, units, x):
    # Rank statistics of a Monte Carlo array: median, IQR and 95% central interval
    p = np.percentile(x, [50, 25, 75, 2.5, 97.5])
    return {'configuration': cfg, 'quantity': name, 'units': units, 'median': p[0], 'p25': p[1],
            'p75': p[2], 'p2.5': p[3], 'p97.5': p[4], 'n_draws': len(x)}

# One row per quantity per configuration, plus the cross-configuration difference
rows = [summarize(cfg, name, u, x) for cfg, res in RES.items() for name, (u, x) in res.items()]
rows.append(summarize('PC_CT10 - PC_CT82', 'F_total (A) difference', 'm3/yr', dF))

# ---- Principal Cordillera parent flux split into Teno (CT-5) and Lontué (CT-6), PC_CT82 only ----
# These are the two terms of F_PC (A) = E5*A5 + E6*A6, kept separate. E5 and E6 are the shared,
# flux-ordered parent draws from section 1 (identical in both configurations), so the split uses
# the same Monte Carlo iterations as every other Route A quantity; nothing is redrawn.
F_CT5 = E5 * A['CT-5']                  # Teno (CT-5) erosional flux, m^3/yr, one value per draw
F_CT6 = E6 * A['CT-6']                  # Lontué (CT-6) erosional flux, m^3/yr, one value per draw
# The two terms must add back to the Route A parent flux on every draw
assert np.allclose(F_CT5 + F_CT6, RES['PC_CT82']['F_PC (A)'][1]), 'CT-5 + CT-6 split does not sum to F_PC'
# Teno share of the parent flux, computed draw by draw (not a ratio of the two medians), as a
# fraction to match the other share_* rows; Table S7 reports it as a percentage
share_CT5 = F_CT5 / (F_CT5 + F_CT6)
# Appended after the existing rows so every pre-existing CSV row keeps its position and values.
# Units follow the CSV convention (m^3/yr, fraction); Table S7 converts to km^3/Myr and %.
for name, u, x in [('F_CT5 = E5*A5 (A)', 'm3/yr', F_CT5),
                   ('F_CT6 = E6*A6 (A)', 'm3/yr', F_CT6),
                   ('share_CT5 of F_PC = F_CT5/(F_CT5+F_CT6) (A)', 'fraction', share_CT5)]:
    rows.append(summarize('PC_CT82', name, u, x))
summary = pd.DataFrame(rows)
# The CSV keeps full precision in long format (machine-readable); the table below is for reading
summary.to_csv(os.path.join(OUT, 'mass_balance_summary.csv'), index=False)

### Readable summary

For each configuration: median, IQR (25th to 75th percentile) and 95% range (2.5th to 97.5th
percentile). Fluxes are in m³/yr, rounded to 3 significant figures; rates are in m/Myr. Rows are
grouped by route. Full precision is in `mass_balance_summary.csv`.

In [7]:
# Display layout: (section, readable label, quantity key in RES, units)
LAYOUT = [
    ('Route A: parent sum', 'PC parents flux, E5·A5 + E6·A6', 'F_PC (A)', 'm3/yr'),
    ('Route A: parent sum', 'Central Depression flux', 'F_CD (A)', 'm3/yr'),
    ('Route A: parent sum', 'CT-3* flux, E3·A_CC', 'F_CC (A)', 'm3/yr'),
    ('Route A: parent sum', 'TOTAL flux at CT-9', 'F_total (A)', 'm3/yr'),
    ('Route A: parent sum', 'Basin rate, F_total / A(CT-9)', 'E_mass (A)', 'm/Myr'),
    ('Route A: parent sum', 'PC rate, F_PC / (A5 + A6)', 'E_PC_mass = F_PC/(A5+A6) (A)', 'm/Myr'),
    ('Route A: parent sum', 'Share of total: PC parents', 'share_PC (A)', 'fraction'),
    ('Route A: parent sum', 'Share of total: CT-3*', 'share_CC (A)', 'fraction'),
    ('Route B: fertility check', 'Q_Teno / Q_Lontue (step 1)', 'q1 = Q_Teno/Q_Lontue (B)', 'ratio'),
    ('Route B: fertility check', 'TOTAL flux at CT-9', 'F_total (B)', 'm3/yr'),
    ('Route B: fertility check', 'Per-draw (B − A) / A', 'rel_diff (B-A)/A', 'tiny'),
    ('Single-sample budgets', 'S1: E9·A9 (outlet only)', 'S1 = E9*A9', 'm3/yr'),
    ('Single-sample budgets', 'S1 as a rate', 'S1 rate', 'm/Myr'),
    ('Single-sample budgets', 'S1 / Route A total', 'S1/F_total', 'ratio'),
    ('Single-sample budgets', 'S2: E_PC·A_PC + E3·A_CC', 'S2 = E_PC*A_PC + E3*A_CC', 'm3/yr'),
    ('Single-sample budgets', 'S2 as a rate', 'S2 rate', 'm/Myr'),
    ('Single-sample budgets', 'S2 / Route A total', 'S2/F_total', 'ratio'),
]

# Number format for each unit type, chosen so each value shows only meaningful digits
FMT = {'m3/yr': lambda v: f'{float(f"{v:.3g}"):,.0f}',  # 3 significant figures, thousands separators
       'm/Myr': lambda v: f'{v:.1f}',                   # rates to 0.1 m/Myr
       'fraction': lambda v: f'{100 * v:.1f}%',         # shares as percentages
       'ratio': lambda v: f'{v:.2f}',                   # dimensionless ratios
       'tiny': lambda v: f'{v:.0e}'}                    # floating-point-level differences

def cells(x, units):
    # Three display strings for one Monte Carlo array: median, IQR, 2.5-97.5% range
    f = FMT[units]
    p = np.percentile(x, [50, 25, 75, 2.5, 97.5])
    return f(p[0]), f'{f(p[1])} to {f(p[2])}', f'{f(p[3])} to {f(p[4])}'

STATS = ['median', 'IQR', '95% range']
UNIT_LABEL = {'m3/yr': 'm³/yr', 'm/Myr': 'm/Myr', 'fraction': '%', 'ratio': '–', 'tiny': '–'}
records = []
for sec, lab, key, u in LAYOUT:
    rec = {('', 'units'): UNIT_LABEL[u]}
    for cfg in CONFIGS:                                   # one block of three columns per configuration
        rec.update(zip([(cfg, s) for s in STATS], cells(RES[cfg][key][1], u)))
    records.append(rec)
readable = pd.DataFrame(records, index=pd.MultiIndex.from_tuples(
    [(sec, lab) for sec, lab, _, _ in LAYOUT], names=['section', 'quantity']))  # group rows by route
readable.columns = pd.MultiIndex.from_tuples(readable.columns)                  # configuration over stat

display(readable)   # renders as an HTML table in Jupyter

# Cross-configuration difference on its own line (it belongs to neither configuration)
med, iqr, ci = cells(dF, 'm3/yr')
print(f'F_total(PC_CT10) − F_total(PC_CT82): median {med} m³/yr, IQR {iqr}, 95% range {ci}')

PC_CT10  \
                                                         units     median   
section                  quantity                                           
Route A: parent sum      PC parents flux, E5·A5 + E6·A6  m³/yr    695,000   
                         Central Depression flux         m³/yr          0   
                         CT-3* flux, E3·A_CC             m³/yr     39,100   
                         TOTAL flux at CT-9              m³/yr    735,000   
                         Basin rate, F_total / A(CT-9)   m/Myr      118.7   
                         PC rate, F_PC / (A5 + A6)       m/Myr      170.9   
                         Share of total: PC parents          %      94.7%   
                         Share of total: CT-3*               %       5.3%   
Route B: fertility check Q_Teno / Q_Lontue (step 1)          –       3.50   
                         TOTAL flux at CT-9              m³/yr    735,000   
                         Per-draw (B − A) / A                –      0e+00   
Single-sample budgets    S1: E9·A9 (outlet only)         m³/yr    650,000   
                         S1 as a rate                    m/Myr      105.0   
                         S1 / Route A total                  –       0.88   
                         S2: E_PC·A_PC + E3·A_CC         m³/yr  1,240,000   
                         S2 as a rate                    m/Myr      199.8   
                         S2 / Route A total                  –       1.68   

                                                                                 \
                                                                            IQR   
section                  quantity                                                 
Route A: parent sum      PC parents flux, E5·A5 + E6·A6      668,000 to 723,000   
                         Central Depression flux                         0 to 0   
                         CT-3* flux, E3·A_CC                   36,900 to 41,200   
                         TOTAL flux at CT-9                  707,000 to 762,000   
                         Basin rate, F_total / A(CT-9)           114.2 to 123.1   
                         PC rate, F_PC / (A5 + A6)               164.2 to 177.6   
                         Share of total: PC parents              94.3% to 95.0%   
                         Share of total: CT-3*                     5.0% to 5.7%   
Route B: fertility check Q_Teno / Q_Lontue (step 1)                2.68 to 4.80   
                         TOTAL flux at CT-9                  707,000 to 762,000   
                         Per-draw (B − A) / A                   -2e-16 to 2e-16   
Single-sample budgets    S1: E9·A9 (outlet only)             614,000 to 686,000   
                         S1 as a rate                             99.2 to 110.8   
                         S1 / Route A total                        0.83 to 0.95   
                         S2: E_PC·A_PC + E3·A_CC         1,170,000 to 1,310,000   
                         S2 as a rate                            188.5 to 211.0   
                         S2 / Route A total                        1.57 to 1.80   

                                                                                 \
                                                                      95% range   
section                  quantity                                                 
Route A: parent sum      PC parents flux, E5·A5 + E6·A6      617,000 to 777,000   
                         Central Depression flux                         0 to 0   
                         CT-3* flux, E3·A_CC                   32,700 to 45,400   
                         TOTAL flux at CT-9                  655,000 to 816,000   
                         Basin rate, F_total / A(CT-9)           105.9 to 131.9   
                         PC rate, F_PC / (A5 + A6)               151.6 to 190.9   
                         Share of total: PC parents              93.6% to 95.6%   
                         Share of 

F_total(PC_CT10) − F_total(PC_CT82): median 2,520 m³/yr, IQR 2,380 to 2,660, 95% range 2,110 to 2,930


### Key findings: scenarios and PC samples compared

One row per flux scenario. For each PC sample the table shows the total flux at CT-9 (median and 95%
range), the same flux as a basin-wide rate (÷ A(CT-9)), and its size relative to the Route A total.
The last column is the change in the median flux from switching PC = CT-8.2 to PC = CT-10.

The Route B change is smaller than Route A's (about 1,800 vs 2,520 m³/yr). This is only because
Route B keeps a slightly different set of draws for each PC sample (99.8% vs 98.4%). On any shared
draw, Route B equals Route A exactly.

In [8]:
# Scenario label -> (key of the total-flux array in RES, key of its per-draw ratio to Route A)
SCEN = {
    'Route A: CT-5 + CT-6 + CT-3* (primary)': ('F_total (A)', None),
    'Route B: quartz-fertility check':        ('F_total (B)', 'rel_diff (B-A)/A'),
    'S1: CT-9 alone':                         ('S1 = E9*A9', 'S1/F_total'),
    'S2: PC sample + CT-3*':                  ('S2 = E_PC*A_PC + E3*A_CC', 'S2/F_total'),
}

def vs_route_a(res, rkey):
    # Median size of a scenario relative to Route A, per draw
    if rkey is None:
        return '1.00 (reference)'              # Route A is the reference
    r = res[rkey][1]
    if rkey.startswith('rel_diff'):            # Route B stores (B - A)/A, so add 1 to get B/A
        return f'{1 + np.median(r):.2f} (identical)'
    return f'{np.median(r):.2f} ({np.percentile(r, 2.5):.2f} to {np.percentile(r, 97.5):.2f})'

f3 = FMT['m3/yr']                               # 3-significant-figure flux format from above
key_rows = []
for scen, (fkey, rkey) in SCEN.items():
    row = {}
    for cfg, pc_label in [('PC_CT10', 'PC = CT-10'), ('PC_CT82', 'PC = CT-8.2')]:
        F = RES[cfg][fkey][1]                   # per-draw total flux for this scenario
        row[(pc_label, 'flux, m³/yr')] = f3(np.median(F))
        row[(pc_label, '95% range')] = f'{f3(np.percentile(F, 2.5))} to {f3(np.percentile(F, 97.5))}'
        row[(pc_label, 'rate, m/Myr')] = f"{np.median(F) / A['CT-9']:.1f}"   # flux spread over CT-9
        row[(pc_label, '× Route A')] = vs_route_a(RES[cfg], rkey)
    # Difference of the medians between the two PC samples (S1 does not use PC, so it is 0)
    d = np.median(RES['PC_CT10'][fkey][1]) - np.median(RES['PC_CT82'][fkey][1])
    row[('CT-10 − CT-8.2', 'Δ median flux, m³/yr')] = f'{float(f"{d:.3g}"):+,.0f}' if d else '0'
    key_rows.append(row)

key_findings = pd.DataFrame(key_rows, index=pd.Index(list(SCEN), name='scenario'))
key_findings.columns = pd.MultiIndex.from_tuples(key_findings.columns)   # PC sample over metric
display(key_findings)   # renders as an HTML table in Jupyter

PC = CT-10                          \
                                       flux, m³/yr               95% range   
scenario                                                                     
Route A: CT-5 + CT-6 + CT-3* (primary)     735,000      655,000 to 816,000   
Route B: quartz-fertility check            735,000      656,000 to 817,000   
S1: CT-9 alone                             650,000      546,000 to 754,000   
S2: PC sample + CT-3*                    1,240,000  1,030,000 to 1,440,000   

                                                                         \
                                       rate, m/Myr            × Route A   
scenario                                                                  
Route A: CT-5 + CT-6 + CT-3* (primary)       118.7     1.00 (reference)   
Route B: quartz-fertility check              118.7     1.00 (identical)   
S1: CT-9 alone                               105.0  0.88 (0.72 to 1.07)   
S2: PC sample + CT-3*                        199.8  1.68 (1.37 to 2.04)   

                                       PC = CT-8.2                          \
                                       flux, m³/yr               95% range   
scenario                                                                     
Route A: CT-5 + CT-6 + CT-3* (primary)     732,000      653,000 to 814,000   
Route B: quartz-fertility check            733,000      657,000 to 814,000   
S1: CT-9 alone                             650,000      546,000 to 754,000   
S2: PC sample + CT-3*                    1,340,000  1,120,000 to 1,560,000   

                                                                         \
                                       rate, m/Myr            × Route A   
scenario                                                                  
Route A: CT-5 + CT-6 + CT-3* (primary)       118.3     1.00 (reference)   
Route B: quartz-fertility check              118.4     1.00 (identical)   
S1: CT-9 alone                               105.0  0.89 (0.73 to 1.07)   
S2: PC sample + CT-3*                        216.3  1.83 (1.48 to 2.22)   

                                             CT-10 − CT-8.2  
                                       Δ median flux, m³/yr  
scenario                                                     
Route A: CT-5 + CT-6 + CT-3* (primary)               +2,520  
Route B: quartz-fertility check                      +1,830  
S1: CT-9 alone                                            0  
S2: PC sample + CT-3*                              -102,000

## 8. Table S7

Written to `supplemental/Table_S7.xlsx`, one sheet, with a header row for each block so that every
column header carries its units.

- **Part A, sediment budget** (PC_CT82 configuration, Route A draws from section 3-5): areas, fluxes,
  rates and shares of the sediment flux, as median and IQR (25th to 75th percentile). Fluxes are in
  km³/Myr (1 km³/Myr = 1000 m³/yr). The PC rate uses the parent area A(CT-5) + A(CT-6).
  The parent flux is also given split into Teno (CT-5) and Lontué (CT-6), and the Teno share of the
  parent flux is computed per draw as 100 · F_CT-5 / (F_CT-5 + F_CT-6).
- **Part B, mineral flux ratios** (parent A / parent B), read from notebook 06:
  `quartz_fertility_and_flux.csv` (`Qflux_lin_*`) and `zircon_fertility_and_flux_by_metric.csv`
  (`Zflux_lin_*`, V_max). The share of parent A is F / (1 + F); this is monotonic in F, so the share of
  the median (and of each quartile) of F is the median (and quartile) of the share.


In [9]:
from openpyxl import Workbook   # writes the .xlsx, like the other supplemental Table_S*.xlsx files

# ---- Part A: sediment budget, PC_CT82 configuration ----
res = RES['PC_CT82']                       # per-draw Route A arrays for PC = CT-8.2
A_par, A_CD, A_CC = AREAS['PC_CT82']       # parent, Central Depression and CT-3* areas (km^2)
KM3 = 1e-3                                 # 1 km^3/Myr = 1e9 m^3 / 1e6 yr = 1000 m^3/yr, so m^3/yr x 1e-3 = km^3/Myr
STAT = ['median', '25th percentile', '75th percentile']   # median and IQR bounds, in column order

def q3(x):
    # Median, 25th and 75th percentile of one Monte Carlo array, rounded to 0.1
    return [round(float(v), 1) for v in np.percentile(x, [50, 25, 75])]

rows_s7 = [['Part A. Sediment budget at CT-9 (PC = CT-8.2 configuration)']]   # rows written top to bottom
# Areas are fixed (no draws), so they have a single value each
rows_s7.append(['Area', 'Area (km2)', 'Area (% of A_CT-9)'])
for lab, a in [('Principal Cordillera parents (A_CT-5 + A_CT-6)', A_par),
               ('Central Depression (A_CT-8.2 - A_CT-5 - A_CT-6)', A_CD),
               ('CT-3* (A_CT-9 - A_CT-8.2)', A_CC)]:
    rows_s7.append([lab, round(a, 1), round(100 * a / A['CT-9'], 1)])          # km^2 and % of the outlet area
rows_s7.append([])                                                              # blank row between blocks

# Volumetric sediment fluxes, converted from m^3/yr to km^3/Myr
rows_s7.append(['Sediment flux'] + [f'Flux, {s} (km3/Myr)' for s in STAT])
for lab, x in [('Principal Cordillera parents (E_CT-5 A_CT-5 + E_CT-6 A_CT-6)', res['F_PC (A)'][1]),
               # The two parent terms separately (per-draw arrays from section 7, same draws as F_PC).
               # Medians of the parts need not add exactly to the median of the sum.
               ('Teno, CT-5 (E_CT-5 A_CT-5)', F_CT5),
               ('Lontué, CT-6 (E_CT-6 A_CT-6)', F_CT6),
               ('Central Depression (transfer zone)', res['F_CD (A)'][1]),                 # zero by assumption
               ('CT-3* (E_CT-3 A_CT-3*)', res['F_CC (A)'][1]),
               ('Total at CT-9', res['F_total (A)'][1]),
               ('CT-9 alone (E_CT-9 A_CT-9)', res['S1 = E9*A9'][1]),                         # single-sample budget S1
               ('Total with CT-10 as the PC proxy (PC_CT10)', RES['PC_CT10']['F_total (A)'][1])]:
    rows_s7.append([lab] + q3(x * KM3))
rows_s7.append([])

# Area-normalised erosion rates
rows_s7.append(['Erosion rate'] + [f'Rate, {s} (m/Myr)' for s in STAT])
for lab, x in [('Basin-averaged (F_total / A_CT-9)', res['E_mass (A)'][1]),
               ('Principal Cordillera (F_PC / (A_CT-5 + A_CT-6))', res['E_PC_mass = F_PC/(A5+A6) (A)'][1]),
               ('Coastal Cordillera (F_CT-3* / A_CT-3*)', res['F_CC (A)'][1] / A_CC)]:   # equals the CT-3 rate
    rows_s7.append([lab] + q3(x))
rows_s7.append([])

# Shares of the total sediment flux at CT-9, as percentages
rows_s7.append(['Share of sediment flux'] + [f'Share, {s} (%)' for s in STAT])
for lab, x in [('Principal Cordillera', res['share_PC (A)'][1]),
               ('Coastal Cordillera (CT-3*)', res['share_CC (A)'][1]),
               # Share of the parent flux only (not of the CT-9 total): 100 * F_CT5 / (F_CT5 + F_CT6),
               # evaluated per draw in section 7 before taking the median and IQR
               ('Teno share of Principal Cordillera parent flux', share_CT5)]:
    rows_s7.append([lab] + q3(100 * x))
rows_s7.append([])

# ---- Part B: mineral flux ratios from notebook 06 ----
qz = pd.read_csv(os.path.join(FERT, 'quartz_fertility_and_flux.csv')).set_index('scenario')   # quartz, all draws
zr = pd.read_csv(os.path.join(FERT, 'zircon_fertility_and_flux_by_metric.csv'))
zr = zr[zr['metric'] == 'vmax'].set_index('scenario')                                         # zircon, V_max mixing
PART_B = ['CT-5 & CT-6 to CT-10', 'CT-5 & CT-6 to CT-11', 'CT-5 & CT-6 to CT-8.2',            # PC parents
          'CT-8.2 & CT-3* to CT-2', 'CT-8.2 & CT-3* to CT-9']                                 # CT-8.2 and CT-3*
Q3 = ['median', 'p25', 'p75']                                                                  # CSV column suffixes

def sig3(v):
    # Flux ratio rounded to 3 significant figures
    return float(f'{v:.3g}')

def share(F):
    # Share of parent A in the combined flux of parents A and B, F = F_A / F_B, as % to 0.1
    return round(100 * F / (1 + F), 1)

rows_s7.append(['Part B. Mineral flux ratios, parent A / parent B (zircon from V_max mixing)'])
rows_s7.append(['Parents (A / B)', 'Child']
               + [f'Quartz flux ratio, {s}' for s in STAT]
               + [f'Quartz share of parent A, {s} (%)' for s in STAT]
               + [f'Zircon flux ratio, {s}' for s in STAT]
               + [f'Zircon share of parent A, {s} (%)' for s in ['bootstrap median'] + STAT[1:]])
for sc in PART_B:
    Fq = [qz.loc[sc, f'Qflux_lin_{s}'] for s in Q3]          # quartz flux ratio: median, p25, p75
    Fz = [zr.loc[sc, f'Zflux_lin_{s}'] for s in Q3]          # zircon flux ratio: median, p25, p75 (bootstrap)
    parents = qz.loc[sc, 'parents'].replace(' / CT-3', ' / CT-3*')   # CT-3 stands for the CT-3* area here
    rows_s7.append([parents, qz.loc[sc, 'child']]
                   + [sig3(F) for F in Fq] + [share(F) for F in Fq]
                   + [sig3(F) for F in Fz] + [share(F) for F in Fz])

wb = Workbook()                       # new workbook, one sheet
ws = wb.active
ws.title = 'Table S7'
for r in rows_s7:
    ws.append(r)                      # [] appends a blank separator row
wb.save(os.path.join(SUPP, 'Table_S7.xlsx'))

# Print the table as written
for r in rows_s7:
    print(' | '.join(str(v) for v in r))


Part A. Sediment budget at CT-9 (PC = CT-8.2 configuration)
Area | Area (km2) | Area (% of A_CT-9)
Principal Cordillera parents (A_CT-5 + A_CT-6) | 4069.1 | 65.7
Central Depression (A_CT-8.2 - A_CT-5 - A_CT-6) | 881.3 | 14.2
CT-3* (A_CT-9 - A_CT-8.2) | 1239.3 | 20.0

Sediment flux | Flux, median (km3/Myr) | Flux, 25th percentile (km3/Myr) | Flux, 75th percentile (km3/Myr)
Principal Cordillera parents (E_CT-5 A_CT-5 + E_CT-6 A_CT-6) | 695.5 | 668.0 | 722.8
Teno, CT-5 (E_CT-5 A_CT-5) | 618.8 | 591.7 | 645.8
Lontué, CT-6 (E_CT-6 A_CT-6) | 76.7 | 72.5 | 80.9
Central Depression (transfer zone) | 0.0 | 0.0 | 0.0
CT-3* (E_CT-3 A_CT-3*) | 36.5 | 34.5 | 38.6
Total at CT-9 | 732.1 | 704.4 | 759.5
CT-9 alone (E_CT-9 A_CT-9) | 650.0 | 614.1 | 686.0
Total with CT-10 as the PC proxy (PC_CT10) | 734.6 | 706.9 | 762.0

Erosion rate | Rate, median (m/Myr) | Rate, 25th percentile (m/Myr) | Rate, 75th percentile (m/Myr)
Basin-averaged (F_total / A_CT-9) | 118.3 | 113.8 | 122.7
Principal Cordillera (F_PC 

## 9. Figures (commented out; uncomment to save to `figures/mass_balance/`)

In [10]:
# import matplotlib.pyplot as plt
# os.makedirs(FIG, exist_ok=True)                       # create the figure directory on demand
# for cfg, res in RES.items():                          # one figure per configuration
#     F_total, S1, S2 = res['F_total (A)'][1], res['S1 = E9*A9'][1], res['S2 = E_PC*A_PC + E3*A_CC'][1]
#     fig, ax = plt.subplots(figsize=(6, 4))
#     bins = np.linspace(0, np.percentile(S2, 99.5), 120)  # common bins spanning all three budgets
#     for x, lab in [(F_total, 'Route A: parent sum'), (S1, 'S1: outlet only'), (S2, 'S2: PC + CT-3*')]:
#         ax.hist(x, bins=bins, histtype='step', density=True, label=lab)
#     ax.set_xlabel('Volumetric sediment flux at CT-9 (m$^3$/yr)')
#     ax.set_ylabel('Probability density')
#     ax.set_title(cfg)
#     ax.legend(frameon=False)
#     fig.tight_layout()
#     fig.savefig(os.path.join(FIG, f'ct9_flux_budgets_{cfg}.pdf'))

In [11]:
# import matplotlib.pyplot as plt
# os.makedirs(FIG, exist_ok=True)
# for cfg, res in RES.items():
#     shares = [res[k][1] for k in ('share_PC (A)', 'share_CD (A)', 'share_CC (A)')]
#     med = [np.median(s) for s in shares]                                        # median shares
#     err = [[m - np.percentile(s, 2.5) for m, s in zip(med, shares)],
#            [np.percentile(s, 97.5) - m for m, s in zip(med, shares)]]
#     fig, ax = plt.subplots(figsize=(4, 4))
#     ax.bar(['PC', 'CD', 'CT-3*'], med, yerr=err, capsize=4)
#     ax.set_ylabel('Share of F$_{total}$ at CT-9')
#     ax.set_title(cfg)
#     fig.tight_layout()
#     fig.savefig(os.path.join(FIG, f'ct9_domain_shares_{cfg}.pdf'))

## Assumptions

1. **Two configurations.** The whole calculation was run with the downstream Principal Cordillera
   reference sample PC = CT-10 (`PC_CT10`) and PC = CT-8.2 (`PC_CT82`). They differ **only** in
   whether the 85.5 km² reach between CT-10 and CT-8.2 is classified as Central Depression (zero
   flux) or as Coastal Cordillera (CT-3* at the CT-3 rate). The Route A totals therefore differ by
   exactly E(CT-3) × 85.5 km² per draw.
2. **Central Depression = 0.** The CD (A(PC) − A(CT-5) − A(CT-6)) is treated as a transfer zone with
   no net erosional contribution to the flux.
3. **CT-3 represents the full CT-3\* area.** The CT-3 rate is applied to the whole Coastal
   Cordillera residual A(CT-9) − A(PC), not only to the CT-3 watershed.
4. **Full watershed areas.** Every area is the full delineated watershed. No Central Depression
   adjustment is made.
5. **Flux-ordered draws.** CT-5, CT-6 and the PC sample come from the index-aligned flux-ordered
   triple (manuscript Methods, "Flux-Ordered Erosion Rates"). Both configurations share the same
   CT-5/CT-6 draws. CT-3 and CT-9 are unconstrained and paired independently.
6. **Uncertainty scope.** The reported intervals propagate **erosion-rate uncertainty only**. They do
   not include sample representativeness (e.g. how well CT-3 stands for CT-3\*) or the different
   ¹⁰Be integration timescales of the samples.